###Build Races Dimension

1. Read silver races table
2. Read silver circuits table
3. Join the data from races with circuits using circuit_id
4. Select the required columns
    - races.season
    - races.round
    - races.race_name
    - races.race_date
    - circuits.circuit_name
    - circuits.locality
    - circuits.country
5. Write the transformed data to gold dim_races table

In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_races"

##### Step 1 - Read source tables
    - circuits
    - races

In [0]:
circuits_df = spark.table(f"{catalog_name}.{silver_schema}.circuits")
races_df = spark.table(f"{catalog_name}.{silver_schema}.races")

##### Step 2 - Join races with circuits using circuit_id
Select the following columns   
    1. races.season   
    2. races.round   
    3. races.race_name   
    4. races.race_date   
    5. circuits.circuit_name   
    6. circuits.locality   
    7. circuits.country   

In [0]:
dim_races_df = (
    races_df
        .join(
            circuits_df, 
            races_df.circuit_id == circuits_df.circuit_id,
            "inner"
            )
        .select(
            races_df.season,
            races_df.round,
            races_df.race_name,
            races_df.race_date,
            circuits_df.circuit_name,
            circuits_df.locality,
            circuits_df.country
        )
)

In [0]:
display(dim_races_df)

##### Step 3 - Write the transformed data to the gold dim_races table

In [0]:
(
    dim_races_df
        .write
        .format("delta")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))